#Teaching AI to Play: From Supervised Imitation to Reinforcement Learning

In this notebook, we will use two approaches to play Tic-Tac-Toe. Our goal is to understand the shift between traditional Machine Learning (learning from static data) and Reinforcement Learning (learning through dynamic experience).

By the end of this exercise, you will see firsthand why games and robotics often require algorithms that can explore and adapt, rather than just memorize.

##Part 1: The Imitator (Supervised Machine Learning)

The Approach: We will train a Random Forest Classifier using an "expert dataset" of pre-recorded board states and optimal moves.

The Concept: The algorithm learns by imitating the data. It treats playing a game exactly like classifying an image or predicting a house price—mapping inputs (the board) to outputs (the move) without understanding the rules of the game.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def generate_expert_data(num_samples=5000):
    X = []
    y = []
    for _ in range(num_samples):
        board = np.random.choice([0, 1, -1], size=9) # 0 means empty space, 1 and -1 mean positions taken by the Player 1 and Player 2 respectively
        empty_spots = np.where(board == 0)[0]

        if len(empty_spots) == 0:
            continue

        target_move = np.random.choice(empty_spots)

        # Simulate an "expert" by prioritizing the center, then corners, then edges
        if 4 in empty_spots:
            target_move = 4
        else:
            corners = [0, 2, 6, 8]
            available_corners = [c for c in corners if c in empty_spots]
            if available_corners:
                target_move = np.random.choice(available_corners)

        X.append(board)
        y.append(target_move)

    return np.array(X), np.array(y)

X, y = generate_expert_data(10000)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.2f}")

sample_board = np.array([[1, -1, 0,
                          0, 1, -1,
                          0, 0, 0]])

predicted_move = model.predict(sample_board)
predicted_probabilities = model.predict_proba(sample_board)

print(f"Board State: \n{sample_board[0].reshape(3,3)}")
print(f"Predicted Move Index: {predicted_move[0]}")
print(f"Move Probabilities: \n{predicted_probabilities[0].reshape(3,3)}")

Model Accuracy: 0.81
Board State: 
[[ 1 -1  0]
 [ 0  1 -1]
 [ 0  0  0]]
Predicted Move Index: 8
Move Probabilities: 
[[0.03666667 0.         0.31633333]
 [0.         0.01       0.        ]
 [0.28866667 0.         0.34833333]]


Guiding Questions:

*   Examine the generate_expert_data function. Does the algorithm actually understand the rules of Tic-Tac-Toe, or is it learning something else?
*   If we change the generate_expert_data function to always choose the lowest available index (0, then 1, then 2...), what strategy will the trained model learn?
*   What happens if the RandomForestClassifier predicts an index (like 4) but that square is already occupied by a -1 on the actual board? How would you handle this in a real game loop?





##Part 2: The Explorer (Reinforcement Learning with Q-Learning)

The Approach: We will throw away the dataset. Instead, we will define the rules of the game and a reward system (+1 for winning, -1 for losing) and force it to play thousands of games against a random opponent.

The Concept: The algorithm learns by trial and error. Using the Bellman equation, it updates a "Q-table" that tracks the long-term value of every possible move in every possible board state.

The Breakthrough: The algorithm discovers its own winning strategy organically, learning to block opponents and set up traps simply by maximizing its expected reward.

In [ ]:
import numpy as np
import random

# Hyperparameters
alpha = 0.5       # Learning rate
gamma = 0.9       # Discount factor
epsilon = 0.2     # Exploration rate
episodes = 5000

# We use a dictionary for the Q-table to store states only as we encounter them
q_table = {}

def get_state_key(board):
    return str(board.tolist())

def get_available_moves(board):
    return np.where(board == 0)[0]

def check_winner(board):
    lines = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)] # Win positions
    for line in lines:
        if board[line[0]] == board[line[1]] == board[line[2]] != 0:
            return board[line[0]]
    if 0 not in board:
        return "Draw"
    return 0

# Training Loop
for _ in range(episodes):
    board = np.zeros(9, dtype=int)
    done = False

    while not done:
        state = get_state_key(board)
        available_moves = get_available_moves(board)

        # Initialize state in Q-table if it's new, i.e. table of available moves e.g. {2: 0.0, 6: 0.0, 8: 0.0}
        if state not in q_table:
            q_table[state] = {m: 0.0 for m in available_moves}

        # Epsilon-Greedy Action Selection
        if random.uniform(0, 1) < epsilon:
            action = random.choice(available_moves)
        else:
            action = max(q_table[state], key=q_table[state].get)

        # Agent (Player 1) makes a move
        board[action] = 1
        reward = 0
        winner = check_winner(board)

        if winner == 1:
            reward = 1
            done = True
        elif winner == "Draw":
            reward = 0
            done = True
        else:
            # Environment (Player 2) makes a random move
            opp_moves = get_available_moves(board)
            opp_action = random.choice(opp_moves)
            board[opp_action] = -1

            winner = check_winner(board)
            if winner == -1:
                reward = -1
                done = True
            elif winner == "Draw":
                reward = 0
                done = True

        # Q-Value Update (Bellman Equation)
        next_state = get_state_key(board)
        next_moves = get_available_moves(board)

        if done:
            max_next_q = 0
        else:
            if next_state not in q_table:
                q_table[next_state] = {m: 0.0 for m in next_moves}
            max_next_q = max(q_table[next_state].values())

        # The Core Q-Learning Update
        current_q = q_table[state][action]
        q_table[state][action] = current_q + alpha * (reward + gamma * max_next_q - current_q)

print(f"Training complete! Learned Q-values for {len(q_table)} states.")

# Test the agent on the exact same board state from Part 1
test_board = np.array([1, -1, 0,
                       0, 1, -1,
                       0, 0, 0])
test_state = get_state_key(test_board)

if test_state in q_table:
    print(f"\nBoard State: \n{test_board.reshape(3,3)}")
    print(f"Q-Values for available moves: {q_table[test_state]}")
    best_move = max(q_table[test_state], key=q_table[test_state].get)
    print(f"Agent's chosen move index: {best_move}")
else:
    print("The agent never encountered this state during training!")

Training complete! Learned Q-values for 1065 states.

Board State: 
[[ 1 -1  0]
 [ 0  1 -1]
 [ 0  0  0]]
Q-Values for available moves: {np.int64(2): 0.1227337646484375, np.int64(3): 0.3290346908569336, np.int64(6): 0.421875, np.int64(7): 0.0, np.int64(8): 0.9999999925494194}
Agent's chosen move index: 8
